In [ ]:
import uuid, warnings
from datetime import datetime
from statistics import NormalDist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

CATALOGO = "desafio_vendas"
G = f"{CATALOGO}.gold"
PASTA = f"/Volumes/{CATALOGO}/gold/graficos"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {G}.graficos")
ID_LOTE = str(uuid.uuid4())

HORIZONTE = 3               # meses a prever
NIVEL_SERVICO = 0.95        # para o estoque de segurança
MIN_TREINO = 24             # histórico mínimo no 1º corte do backtest
TOLERANCIA_WAPE_PP = 0.5    # modelos a até 0,5 p.p. do melhor são considerados equivalentes

serie = (spark.table(f"{G}.agg_vendas_mensal_produto")
         .select("primeiro_dia_mes", "produto", F.col("unidades_demanda").cast("double").alias("y"))
         .toPandas())
serie["primeiro_dia_mes"] = pd.to_datetime(serie["primeiro_dia_mes"])
y = serie.pivot(index="primeiro_dia_mes", columns="produto", values="y").sort_index()
assert y.shape[0] == 36 and not y.isna().any().any(), "Série incompleta"
print(y.index.min().date(), "→", y.index.max().date(), "|", list(y.columns))

In [ ]:
def acf(x, lag):
    x = np.asarray(x, float) - np.mean(x)
    return float(np.sum(x[lag:] * x[:-lag]) / np.sum(x * x))

t = np.arange(len(y))
resumo = pd.DataFrame({
    "media_mensal": y.mean(),
    "desvio": y.std(),
    "cv_pct": 100 * y.std() / y.mean(),
    "variacao_36m_pct": pd.Series({c: 100 * np.polyfit(t, y[c].values, 1)[0] * (len(y) - 1) / y[c].mean() for c in y.columns}),
    "acf_lag1": pd.Series({c: acf(y[c].values, 1) for c in y.columns}),
    "acf_lag12": pd.Series({c: acf(y[c].values, 12) for c in y.columns}),
}).round(3)
display(resumo)
print(f"Banda de ruído branco (~95%): ±{2 / np.sqrt(len(y)):.2f}. "
      "Autocorrelações dentro da banda indicam ausência de dependência temporal relevante.")

In [ ]:
def m_media_historica(h, H): return np.repeat(h.mean(), H)
def m_media_movel_12(h, H):  return np.repeat(h[-12:].mean(), H)
def m_media_movel_6(h, H):   return np.repeat(h[-6:].mean(), H)
def m_media_movel_3(h, H):   return np.repeat(h[-3:].mean(), H)
def m_naive(h, H):           return np.repeat(h[-1], H)
def m_naive_sazonal(h, H):   return np.array([h[-12 + (i % 12)] for i in range(H)])

def m_suavizacao_exp(h, H, alpha=0.3):
    nivel = h[0]
    for x in h[1:]:
        nivel = alpha * x + (1 - alpha) * nivel
    return np.repeat(nivel, H)

def m_tendencia_linear(h, H):
    a, b = np.polyfit(np.arange(len(h)), h, 1)
    return a * np.arange(len(h), len(h) + H) + b

MODELOS = {
    "media_historica": m_media_historica, "media_movel_12": m_media_movel_12,
    "media_movel_6": m_media_movel_6, "media_movel_3": m_media_movel_3,
    "naive": m_naive, "suavizacao_exp": m_suavizacao_exp,
    "naive_sazonal": m_naive_sazonal, "tendencia_linear": m_tendencia_linear,
}
try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    def m_holt_winters(h, H):
        return ExponentialSmoothing(h, trend="add", seasonal="add", seasonal_periods=12).fit().forecast(H)
    MODELOS["holt_winters"] = m_holt_winters
except ImportError:
    print("statsmodels indisponível: Holt-Winters fora da comparação.")

# Backtest com origem expansiva: treina até o corte e prevê os 3 meses seguintes
registros = []
for produto in y.columns:
    s = y[produto].values
    for corte in range(MIN_TREINO, len(s) - HORIZONTE + 1):
        treino, real = s[:corte], s[corte:corte + HORIZONTE]
        for nome, f in MODELOS.items():
            try:
                pred = np.asarray(f(treino, HORIZONTE), float)
            except Exception:
                continue
            for h_, (r, p_) in enumerate(zip(real, pred), start=1):
                registros.append((produto, nome, corte, h_, r, p_))

bt = pd.DataFrame(registros, columns=["produto", "modelo", "corte", "horizonte", "real", "previsto"])
bt["erro"] = bt["previsto"] - bt["real"]
bt["erro_abs"] = bt["erro"].abs()
bt["erro_sq"] = bt["erro"] ** 2
print(f"{bt['corte'].nunique()} cortes × {HORIZONTE} meses × {y.shape[1]} produtos × {bt['modelo'].nunique()} modelos")

In [ ]:
g = bt.groupby("modelo").agg(mae=("erro_abs", "mean"), mse=("erro_sq", "mean"),
                              soma_abs=("erro_abs", "sum"), soma_real=("real", "sum"), soma_erro=("erro", "sum"))
ranking = pd.DataFrame({"MAE": g["mae"], "RMSE": np.sqrt(g["mse"]),
                        "WAPE_pct": 100 * g["soma_abs"] / g["soma_real"],
                        "vies_pct": 100 * g["soma_erro"] / g["soma_real"]}).sort_values("WAPE_pct")

ordem_simplicidade = ["media_historica", "media_movel_12", "media_movel_6", "media_movel_3", "naive",
                      "suavizacao_exp", "naive_sazonal", "tendencia_linear", "holt_winters"]
melhor = ranking["WAPE_pct"].min()
equivalentes = ranking[ranking["WAPE_pct"] <= melhor + TOLERANCIA_WAPE_PP].index.tolist()
escolhido = sorted(equivalentes, key=ordem_simplicidade.index)[0]

ranking["status"] = ["ESCOLHIDO" if m == escolhido else ("equivalente" if m in equivalentes else "pior")
                     for m in ranking.index]
display(ranking.round(2))
print(f"Melhor WAPE: {melhor:.2f}% | equivalentes (±{TOLERANCIA_WAPE_PP} p.p.): {equivalentes} | escolhido: {escolhido}")

cor = {"ESCOLHIDO": "tab:green", "equivalente": "tab:blue", "pior": "lightgray"}
fig, ax = plt.subplots()
ax.barh(ranking.index[::-1], ranking["WAPE_pct"][::-1], color=[cor[s] for s in ranking["status"][::-1]])
ax.set_title("Backtest: erro percentual ponderado (WAPE) por modelo, menor é melhor")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.1f}%"))
fig.tight_layout(); fig.savefig(f"{PASTA}/13_backtest_modelos.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
res = bt[bt["modelo"] == escolhido].assign(residuo=lambda d: -d["erro"])   # real - previsto
datas_prev = pd.date_range(y.index[-1] + pd.offsets.MonthBegin(1), periods=HORIZONTE, freq="MS")
z80 = NormalDist().inv_cdf(0.90)              # intervalo bilateral de 80%
z = NormalDist().inv_cdf(NIVEL_SERVICO)
agora = datetime.now()

def sigma_erro(produto):
    """Desvio do erro de previsão mensal."""
    h = y[produto].values
    if escolhido == "media_historica":
        return float(h.std(ddof=1) * np.sqrt(1 + 1 / len(h)))     # erro de prever com a média da amostra
    r = res.loc[res["produto"] == produto, "residuo"].values
    return float(np.sqrt(np.mean(r ** 2)))                         # RMSE do backtest

prev_rows, est_rows = [], []
for produto in y.columns:
    pred = np.maximum(np.asarray(MODELOS[escolhido](y[produto].values, HORIZONTE), float), 0)
    sigma = sigma_erro(produto)
    for d, p_ in zip(datas_prev, pred):
        prev_rows.append((produto, d.date(), round(float(p_), 1), round(float(max(p_ - z80 * sigma, 0)), 1),
                          round(float(p_ + z80 * sigma), 1), escolhido, ID_LOTE, agora))
    demanda = float(pred.sum())
    seguranca = float(z * sigma * np.sqrt(HORIZONTE))   # assume erros mensais independentes
    est_rows.append((produto, datas_prev[0].date(), datas_prev[-1].date(), round(demanda, 1),
                     round(seguranca, 1), float(np.ceil(demanda + seguranca)), NIVEL_SERVICO,
                     round(sigma, 1), escolhido, ID_LOTE, agora))

prev_df = pd.DataFrame(prev_rows, columns=["produto", "mes_referencia", "unidades_previstas",
                                           "limite_inferior_80", "limite_superior_80", "modelo", "id_lote", "data_processamento"])
est_df = pd.DataFrame(est_rows, columns=["produto", "periodo_inicio", "periodo_fim", "demanda_prevista_3m",
                                         "estoque_seguranca", "estoque_recomendado", "nivel_servico",
                                         "desvio_erro_mensal", "modelo", "id_lote", "data_processamento"])
display(prev_df); display(est_df)

# Conferência: com a média como modelo, ~80% dos meses históricos devem cair dentro do intervalo
if escolhido == "media_historica":
    for produto in y.columns:
        mu, sg = y[produto].mean(), sigma_erro(produto)
        cob = ((y[produto] >= mu - z80 * sg) & (y[produto] <= mu + z80 * sg)).mean()
        print(f"{produto}: {100 * cob:.0f}% dos meses históricos dentro do intervalo de 80%")

In [ ]:
fig, axes = plt.subplots(len(y.columns), 1, figsize=(10, 3.2 * len(y.columns)), sharex=True)
for ax, produto in zip(np.atleast_1d(axes), y.columns):
    p = prev_df[prev_df["produto"] == produto]
    datas = pd.to_datetime(p["mes_referencia"])
    ax.plot(y.index, y[produto], marker="o", markersize=3, label="Histórico")
    ax.plot(datas, p["unidades_previstas"], "--", marker="o", color="tab:red", label="Previsão")
    ax.fill_between(datas, p["limite_inferior_80"], p["limite_superior_80"], color="tab:red", alpha=0.2, label="Intervalo 80%")
    ax.set_title(f"{produto}: demanda mensal (unidades)"); ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25); ax.legend(loc="lower left", ncol=3)
fig.tight_layout(); fig.savefig(f"{PASTA}/12_previsao_demanda.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
def gravar(pdf, nome, comentario):
    destino = f"{G}.{nome}"
    (spark.createDataFrame(pdf).write.format("delta").mode("overwrite")
          .option("overwriteSchema", True).saveAsTable(destino))
    spark.sql(f"COMMENT ON TABLE {destino} IS '{comentario}'")
    print(destino, spark.table(destino).count(), "linhas")

gravar(prev_df, "previsao_demanda_mensal",
       "Previsao de demanda mensal por produto (aprovadas + pendentes) para os 3 meses seguintes ao historico, com intervalo de 80 por cento")
gravar(est_df, "recomendacao_estoque",
       "Necessidade de estoque para cobrir 3 meses: demanda prevista mais estoque de seguranca. Nao considera saldo atual")
gravar(ranking.reset_index().rename(columns={"index": "modelo", "modelo": "modelo"}), "backtest_modelos",
       "Resultado do backtest por modelo (MAE, RMSE, WAPE, vies) e modelo escolhido")

In [ ]:
import json
wape_escolhido = float(ranking.loc[escolhido, "WAPE_pct"])
dbutils.notebook.exit(json.dumps({
    "id_lote": ID_LOTE, "qtd_lida": int(y.size), "qtd_gravada": int(len(prev_df)),
    "qtd_rejeitada": 0, "qtd_duplicada": 0,
    "wape_pct": round(wape_escolhido, 2), "modelo": escolhido,
    "status": "SUCESSO" if len(prev_df) == HORIZONTE * y.shape[1] else "DIVERGENCIA"}))